# 01a — Camera Calibration Following the Paper

This section **only** handles camera calibration: finding the chessboard corner points,
optimizing subpixels, sorting the 49 points, then computing the camera parameters.

**Input:** folder containing chessboard images (`calib_XXXX.png`, grayscale 1280x720).
Board: 8x8 grid of 20 mm squares (7x7 inner corner points) + 3 marker circles
(origin, X axis, Y axis).

**Output:**
1. Intrinsic matrix `K` and distortion coefficients `D`
2. Reprojection error (RMSE) per image and overall
3. Visualization of the pipeline stages (a)-(f)
4. `output/camera_params.npz` + `output/reprojection_rmse.csv` — used by
   notebook `02_machine_calibration/machine_calibration.ipynb`

> The execution results of the original notebook were intentionally cleared (Clear All
> Outputs) so the file stays lightweight for GitHub. The original content remains intact in
> `Semester 7/SEMPRO/Code/kalibrasicamera/kalibrasisesuaipaper.ipynb`.


In [ ]:
# PARAMETER CONFIGURATION

# ==========================================================================
# LOAD PATH CONFIGURATION  (single source for all notebooks: config_paths.py)
# All dataset paths are in that file. Change them there, not here.
# ==========================================================================
import os
import sys

_p = os.path.abspath(os.getcwd())
while not os.path.isfile(os.path.join(_p, "config_paths.py")):
    _parent = os.path.dirname(_p)
    if _parent == _p:
        raise FileNotFoundError(
            "config_paths.py not found. Run this notebook from inside the "
            "curated repo folder, or set the KALIBRASI_REPO environment variable to the repo root."
        )
    _p = _parent
sys.path.insert(0, _p)
import config_paths as CFG

CFG.diagnose()
print()

# Pipeline functions (EDLines, screening, subpixel, sorting) live in
# calibration_pipeline.py so they are shared by all notebooks.
from calibration_pipeline import (
    detect_edlines_raw, get_initial_corners_strict_paper,
    optimize_subpixel_gradient, sort_corners_robust_paper,
)


import cv2
import numpy as np
import glob
import os
from scipy.spatial import distance
import matplotlib.pyplot as plt

# ==========================================
# 1. PARAMETER CONFIGURATION
# ==========================================
CHESSBOARD_SIZE = CFG.CHESSBOARD_SIZE
SQUARE_SIZE = CFG.SQUARE_SIZE_MM  # mm
FOLDER_PATH = CFG.CAMERA_CALIB_DATASET
# Alternative dataset (camera calibration from the measurement error session):
# FOLDER_PATH = CFG.resolve("measurement_error/x_camera_calibration")
objp = np.zeros((CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2) * SQUARE_SIZE

vis_images = {}
image_files = []
for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG', '*.JPEG'):
    image_files.extend(glob.glob(os.path.join(FOLDER_PATH, ext)))
image_files = sorted(list(set(image_files)))

if not image_files:
    print("ERROR: No images found.")
else:
    sample_idx = 0 if len(image_files) > 7 else 2 
    sample_img = cv2.imread(image_files[sample_idx])
    sample_gray = cv2.cvtColor(sample_img, cv2.COLOR_BGR2GRAY)
    gray_3c = cv2.cvtColor(sample_gray, cv2.COLOR_GRAY2RGB)
    vis_images['a'] = gray_3c.copy()

In [ ]:
# 2. EDLINES DETECTOR
# ==========================================


sample_lines = detect_edlines_raw(sample_gray)

img_b = gray_3c.copy()
for line in sample_lines:
    cv2.line(img_b, (int(line[0]), int(line[1])), (int(line[2]), int(line[3])), (255, 0, 0), 2)

# ---> MAKE SURE THIS LINE EXISTS TO SAVE THE IMAGE TO CELL 8 <---
vis_images['b'] = img_b 

# Visualization
plt.figure(figsize=(8, 6))
plt.imshow(img_b)
plt.title('(b) Image after straight-line detection by EDLines', loc='left', fontweight='bold')
plt.axis('off')
plt.show()

In [ ]:
# 3. LINE SCREENING & INITIAL CORNERS
# ==========================================








sample_initial, sample_filt_lines = get_initial_corners_strict_paper(sample_gray, sample_lines)

img_c = gray_3c.copy()
for line in sample_filt_lines:
    cv2.line(img_c, (int(line[0]), int(line[1])), (int(line[2]), int(line[3])), (255, 0, 0), 2)
vis_images['c'] = img_c

img_d = gray_3c.copy()
for pt in sample_initial:
    cv2.circle(img_d, (int(pt[0]), int(pt[1])), 6, (255, 0, 0), -1)
vis_images['d'] = img_d

# Visualization
fig, axs = plt.subplots(1, 2, figsize=(16, 6))
axs[0].imshow(img_c)
axs[0].set_title('(c) Result after straight-line screening', loc='left', fontweight='bold')
axs[0].axis('off')

axs[1].imshow(img_d)
axs[1].set_title('(d) Initial coordinate results of corner points', loc='left', fontweight='bold')
axs[1].axis('off')
plt.show()

In [ ]:
# 4. SUBPIXEL OPTIMIZATION
# ==========================================


sample_refined = optimize_subpixel_gradient(sample_gray, sample_initial)

img_e = gray_3c.copy()
if sample_refined is not None:
    for pt in sample_refined:
        # Change: Draw a small blue point, then a larger red circle around it
        cv2.circle(img_e, (int(pt[0]), int(pt[1])), 2, (255, 0, 0), -1) # Small blue point
        cv2.circle(img_e, (int(pt[0]), int(pt[1])), 8, (0, 0, 255), 2)  # Red circle, larger, thicker line, red color
vis_images['e'] = img_e

# Visualization
plt.figure(figsize=(8, 6))
plt.imshow(img_e)
plt.title('(e) Result of corner points subpixel optimization', loc='left', fontweight='bold')
plt.axis('off')
plt.show()

In [ ]:
# 5. ROBUST CORNER SORTING (FIXED ALIGNMENT)
# ==========================================


sample_sorted = sort_corners_robust_paper(sample_refined, sample_gray, CHESSBOARD_SIZE)

img_f = gray_3c.copy()
if sample_sorted is not None:
    for idx, pt in enumerate(sample_sorted): 
        cv2.circle(img_f, (int(pt[0]), int(pt[1])), 2, (255, 0, 0), -1)
        cv2.putText(img_f, str(idx+1), (int(pt[0])+4, int(pt[1])-4), 1, 1.5, (255, 0, 0), 2)
else:
    cv2.putText(img_f, "EDCIRCLES / SORTING FAILED", (50, 100), 1, 2, (255, 255, 0), 2)
vis_images['f'] = img_f

# Visualization
plt.figure(figsize=(8, 6))
plt.imshow(img_f)
plt.title('(f) Result after corner sorting (Corrected Alignment)', loc='left', fontweight='bold')
plt.axis('off')
plt.show()

In [ ]:
# 6. FINAL PIPELINE: LOOP OVER ALL IMAGES & CALIBRATION
# ==========================================
objpoints, imgpoints, valid_filenames = [], [], []
img_shape = None
expected_pts = CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1]

print(f"\nProcessing {len(image_files)} images...")

for idx, fname in enumerate(image_files):
    img = cv2.imread(fname)
    if img is None: continue
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if img_shape is None: img_shape = gray.shape[::-1]
    
    lines = detect_edlines_raw(gray)
    initial_corners, _ = get_initial_corners_strict_paper(gray, lines)
    
    if len(initial_corners) >= expected_pts:
        refined_corners = optimize_subpixel_gradient(gray, initial_corners)
        sorted_corners = sort_corners_robust_paper(refined_corners, gray, CHESSBOARD_SIZE)
        
        if sorted_corners is not None:
            objpoints.append(objp)
            imgpoints.append(sorted_corners.reshape(-1, 1, 2))
            valid_filenames.append(os.path.basename(fname))
            print(f"[{idx+1:02d}/{len(image_files):02d}] OK    - {os.path.basename(fname)}")
        else:
            print(f"[{idx+1:02d}/{len(image_files):02d}] FAILED - {os.path.basename(fname)} (Sorting Error)")
    else:
        print(f"[{idx+1:02d}/{len(image_files):02d}] FAILED - {os.path.basename(fname)} (Not enough initial points)")

# ==========================================
# 7. CALIBRATION & REPROJECTION ERROR (RMSE) EVALUATION 
# ==========================================
if len(objpoints) > 0:
    print(f"Computing camera parameters from {len(objpoints)} valid images...")
    
    # 1. Compute Camera Parameters (Intrinsic & Distortion)
    ret, camera_matrix, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
        objpoints, imgpoints, img_shape, None, None
    )

    print("\n--- 1. INTRINSIC MATRIX (K) ---")
    print("Format: [[fx, 0, cx], [0, fy, cy], [0, 0, 1]]")
    print(np.round(camera_matrix, 4))
    
    print("\n--- 2. DISTORTION COEFFICIENTS ---")
    print("Format: [k1, k2, p1, p2, k3]")
    print(np.round(dist_coeffs.ravel(), 5))

    # 2. Compute Reprojection Error (RMSE) per Image Following Equation 2.6
    print("\n--- 3. REPROJECTION ERROR (RMSE) PER IMAGE ---")
    total_sse = 0       # Overall Sum of Squared Errors
    total_points = 0    # Overall total points (N)
    
    for i in range(len(objpoints)):
        # Reproject the 3D points onto the image using the obtained parameters (p_hat)
        imgpts2, _ = cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], camera_matrix, dist_coeffs)
        
        # Compute the Euclidean distance (L2 Norm) between the observed point (p) and the projection (p_hat)
        # cv2.norm with NORM_L2 gives: sqrt( sum( (p - p_hat)^2 ) )
        norm_l2 = cv2.norm(imgpoints[i], imgpts2, cv2.NORM_L2)
        
        # Square norm_l2 to remove the root, obtaining the sum of squared errors (SSE)
        sse_per_image = norm_l2 ** 2
        n_points_per_image = len(imgpts2)
        
        # Compute RMSE per image: sqrt( SSE / n )
        rmse_per_image = np.sqrt(sse_per_image / n_points_per_image)
        
        # Accumulate to compute the overall mean
        total_sse += sse_per_image
        total_points += n_points_per_image
        
        print(f"{valid_filenames[i]:<25} : {rmse_per_image:.4f} pixels")
        
    # Compute the overall RMSE (Root Mean Square Error) for all images
    overall_rmse = np.sqrt(total_sse / total_points)
    
    # 3. Accuracy Analysis 
    print("\n--- 4. ACCURACY CONCLUSION ---")
    print(f"=> Overall Reprojection Error (RMSE) : {overall_rmse:.4f} pixels")
    
    if overall_rmse < 0.05:
        print("\n[STATUS]: VERY ACCURATE!")
        print("Matches the accuracy target where error < 0.05 pixels.")
    elif overall_rmse < 0.15:
        print("\n[STATUS]: ACCURATE.")
        print("Meets the high accuracy standard for camera calibration.")
    else:
        print("\n[STATUS]: NORMAL.")
        print("Calibration succeeded, but make sure no images in the dataset are blurry or too dark.")

else:
    print("Calibration cancelled: No valid data to calibrate.")

In [ ]:
# 8. DISPLAY PIPELINE VISUALIZATION (SAMPLE IMAGES)
# ==========================================
titles = {
    'a': '(a) Original grayscale image',
    'b': '(b) Image after straight-line detection by EDLines',
    'c': '(c) Result after straight-line screening',
    'd': '(d) Initial coordinate results of corner points',
    'e': '(e) Result of corner points subpixel optimization',
    'f': '(f) Result after corner sorting'
}

print("Displaying the visualization as a Grid (Combined)...")

# --- OPTION 1: GRID SUBPLOT VIEW (HIGHLY RECOMMENDED FOR THE THESIS) ---
fig, axs = plt.subplots(2, 3, figsize=(20, 12))
axs = axs.ravel() # Flatten the axis array so it is easy to loop over

for i, key in enumerate(['a', 'b', 'c', 'd', 'e', 'f']):
    # Convert BGR to RGB for matplotlib if the image is color
    # (Our vis_images images are already in RGB format from Cell 1)
    axs[i].imshow(vis_images[key])
    axs[i].set_title(titles[key], loc='left', fontweight='bold', fontsize=14)
    axs[i].axis('off')

plt.tight_layout()
plt.show()

print("\nDisplaying the visualization individually...")

# --- OPTION 2: INDIVIDUAL VIEW (LIKE THE ORIGINAL CODE) ---
for key in ['a', 'b', 'c', 'd', 'e', 'f']:
    plt.figure(figsize=(8, 6))
    plt.imshow(vis_images[key])
    plt.title(titles[key], loc='left', fontweight='bold')
    plt.axis('off')
    plt.show()

In [ ]:
# --- OPTION 2: INDIVIDUAL VIEW (NO TITLE & FULL SIZE) ---
# Good if you want to save the images one by one without white margins
print("\nDisplaying the visualization individually (Full Frame, No Title)...")

for key in ['a', 'b', 'c', 'd', 'e', 'f']:
    if key in vis_images:
        img = vis_images[key]
        h, w = img.shape[:2]
        
        # Adjust DPI so the figure size matches the image resolution
        dpi = 100
        fig = plt.figure(figsize=(w/dpi, h/dpi))
        
        # add_axes([left, bottom, width, height]) 
        # Using [0,0,1,1] to remove all white margins/padding
        ax = fig.add_axes([0, 0, 1, 1])
        
        ax.imshow(img)
        ax.axis('off')
        plt.show()
    else:
        print(f"Warning: Image for key '{key}' not found in vis_images.")

In [ ]:
# === [EXPORT] CAMERA PARAMETERS FOR MACHINE CALIBRATION ===
# Save the intrinsic matrix and distortion coefficients as CSV so they can be read by
# notebook 02_machine_calibration/machine_calibration.ipynb (and other scripts) without
# re-running camera calibration.
import os
import numpy as np
import pandas as pd

OUTPUT_DIR = os.path.join(CFG.OUTPUT_DIR, "camera_calibration")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Intrinsic matrix K (3x3) -- pure numeric, easy to read with np.loadtxt
np.savetxt(os.path.join(OUTPUT_DIR, "camera_matrix.csv"), camera_matrix,
           delimiter=",", fmt="%.10f")

# 2. Distortion coefficients [k1, k2, p1, p2, k3] -- one row
np.savetxt(os.path.join(OUTPUT_DIR, "dist_coeffs.csv"),
           np.asarray(dist_coeffs).reshape(1, -1), delimiter=",", fmt="%.10f")

# 3. Description as a parameter/value table (for human reading)
row = {"image_width": int(img_shape[0]), "image_height": int(img_shape[1]),
       "fx": camera_matrix[0, 0], "fy": camera_matrix[1, 1],
       "cx": camera_matrix[0, 2], "cy": camera_matrix[1, 2]}
for i, k in enumerate(np.asarray(dist_coeffs).ravel()):
    row[f"d{i + 1}"] = float(k)
pd.DataFrame(sorted(row.items()), columns=["parameter", "value"]).to_csv(
    os.path.join(OUTPUT_DIR, "camera_params.csv"), index=False)

# 4. Reprojection RMSE per image (Equation 2.6 in the manuscript)
rows = []
for _i in range(len(objpoints)):
    _imgpts2, _ = cv2.projectPoints(
        objpoints[_i], rvecs[_i], tvecs[_i], camera_matrix, dist_coeffs)
    _n2 = cv2.norm(imgpoints[_i], _imgpts2, cv2.NORM_L2)
    rows.append((valid_filenames[_i], float(np.sqrt((_n2 ** 2) / len(_imgpts2)))))
df_rmse = pd.DataFrame(rows, columns=["file", "rmse_px"])
df_rmse.to_csv(os.path.join(OUTPUT_DIR, "reprojection_rmse.csv"), index=False)

print(f"Camera parameters saved as CSV in:\n  {OUTPUT_DIR}")
for f in ["camera_matrix.csv", "dist_coeffs.csv", "camera_params.csv", "reprojection_rmse.csv"]:
    print(f"  - {f}")
print(f"\n{len(rows)} valid images | mean reprojection RMSE: "
      f"{df_rmse['rmse_px'].mean():.4f} px (min {df_rmse['rmse_px'].min():.4f}, "
      f"max {df_rmse['rmse_px'].max():.4f})")
